# RoPE：旋转位置编码的数学、实现与长上下文

本 notebook 从二维旋转推导 RoPE，并用 PyTorch 验证范数、相对位置恒等式、batch position 和长上下文 scaling。

## 学习目标

1. 从复数乘法/旋转矩阵推导 RoPE，并解释为什么 QK 点积只依赖相对位移。
2. 正确实现 split-half RoPE，理解它与 interleaved 布局不可在已有 checkpoint 上混用。
3. 跟踪 `[batch, heads, sequence, head_dim]` 上 cos/sin 的广播。
4. 比较 Position Interpolation、NTK-aware、Dynamic NTK、YaRN 等扩长思路。
5. 处理 KV Cache offset、左 padding、packing 与低精度相位问题。


In [ ]:
import math
import torch

torch.manual_seed(11)
torch.set_printoptions(precision=5, sci_mode=False)
print("PyTorch:", torch.__version__)


## 1. 从二维旋转到相对位置

把一对通道看成复数 `z=x0+i*x1`，在位置 `m` 乘 `exp(i*m*theta)`：

\[R(m\theta)=\begin{bmatrix}\cos(m\theta)&-\sin(m\theta)\\\sin(m\theta)&\cos(m\theta)\end{bmatrix}.\]

对位置 m 的 query 和位置 n 的 key：

\[(R_mq)^T(R_nk)=q^TR_m^TR_nk=q^TR_{n-m}k.\]

因此 attention score 只留下相对位移。单个旋转后的 Q/K 仍依赖绝对位置；causal mask 也仍然需要，RoPE 不会自动屏蔽未来。


In [ ]:
def build_inv_freq(rotary_dim, base=10_000.0, dtype=torch.float32):
    assert rotary_dim % 2 == 0
    idx = torch.arange(0, rotary_dim, 2, dtype=dtype)
    return 1.0 / (base ** (idx / rotary_dim))

def rotate_half_split(x):
    # split-half 配对：(0,D/2), (1,D/2+1), ...
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)

def apply_rope_split(x, position_ids, inv_freq, rotary_dim=None):
    # x: [B,H,S,D]，position_ids: [B,S]
    rotary_dim = rotary_dim or x.size(-1)
    x_rot, x_pass = x[..., :rotary_dim], x[..., rotary_dim:]
    freqs = torch.einsum("bs,d->bsd", position_ids.float(), inv_freq.float())
    emb = torch.cat((freqs, freqs), dim=-1)
    cos = emb.cos()[:, None, :, :].to(x.dtype)
    sin = emb.sin()[:, None, :, :].to(x.dtype)
    rotated = x_rot * cos + rotate_half_split(x_rot) * sin
    return torch.cat((rotated, x_pass), dim=-1)

B, H, S, D = 2, 3, 5, 8
x = torch.randn(B, H, S, D)
positions = torch.arange(S).expand(B, -1)
inv_freq = build_inv_freq(D)
x_rot = apply_rope_split(x, positions, inv_freq)
print("x / positions / inv_freq / output:", x.shape, positions.shape, inv_freq.shape, x_rot.shape)
print("最大范数误差:", (x.norm(dim=-1) - x_rot.norm(dim=-1)).abs().max().item())
assert torch.allclose(x.norm(dim=-1), x_rot.norm(dim=-1), atol=1e-5)


### 相对位置恒等式的数值验证

同时把 m、n 平移同一个常数，二者相对距离不变，点积也应不变。下面还直接比较 `(R_m q)·(R_n k)` 和 `q·R_(n-m)k`。


In [ ]:
q = torch.randn(1, 1, 1, D)
k = torch.randn(1, 1, 1, D)

def at_position(vec, pos):
    pid = torch.tensor([[pos]])
    return apply_rope_split(vec, pid, inv_freq)

m, n = 7, 19
lhs = (at_position(q, m) * at_position(k, n)).sum()
rhs = (q * at_position(k, n - m)).sum()
shifted = (at_position(q, m + 100) * at_position(k, n + 100)).sum()
print("lhs, rhs, 同步平移后:", lhs.item(), rhs.item(), shifted.item())
assert torch.allclose(lhs, rhs, atol=2e-5)
assert torch.allclose(lhs, shifted, atol=2e-5)


## 2. 频率、base、rotary dimension 与布局

常见频率为 `theta_i = base^(-2i/dr)`，周期为 `2*pi/theta_i`。前部维对频率高、区分近邻；后部维对频率低、覆盖长距离。`rotary_dim <= head_dim` 且必须为偶数，未旋转维作为 pass-through 内容子空间。

两类配对布局：

- interleaved：`(0,1),(2,3),...`；
- split-half：`(0,D/2),(1,D/2+1),...`。

两者只是维度置换，数学能力等价；但 checkpoint 的 Q/K 权重已适应特定布局，不能只改 `rotate_half`。


In [ ]:
def rotate_half_interleaved(x):
    pairs = x.reshape(*x.shape[:-1], -1, 2)
    y = torch.stack((-pairs[..., 1], pairs[..., 0]), dim=-1)
    return y.flatten(-2)

demo = torch.tensor([1.0, 2.0, 3.0, 4.0])
print("原向量             :", demo)
print("split-half rotate  :", rotate_half_split(demo))
print("interleaved rotate:", rotate_half_interleaved(demo))

periods = 2 * math.pi / build_inv_freq(8)
for i, period in enumerate(periods.tolist()):
    print(f"frequency pair {i}: period={period:.2f} tokens")


## 3. 长上下文 scaling：统一缩放还是分频缩放

原训练窗口之外，公式仍可计算，但模型面对未见过的相位组合。Position Interpolation (PI) 用 `position/factor`，等价于所有频率除以 factor；它避免相位外推，却压缩近距离分辨率。常见 NTK-aware base 变换使最高频近似不变、低频拉伸更多：

\[base'=base\cdot factor^{d_r/(d_r-2)}.\]

Dynamic NTK 在原窗口内保持原频率，越界后按当前长度调节；YaRN/较新分频方法对高、中、低频分别保留、平滑过渡与缩放，并可能调整 attention temperature。LongRoPE 进一步搜索逐维非均匀因子。具体字段和公式存在实现差异，必须跟 checkpoint 官方配置。


In [ ]:
def linear_pi_inv_freq(rotary_dim, base, factor):
    return build_inv_freq(rotary_dim, base) / factor

def ntk_aware_inv_freq(rotary_dim, base, factor):
    assert rotary_dim > 2
    scaled_base = base * factor ** (rotary_dim / (rotary_dim - 2))
    return build_inv_freq(rotary_dim, scaled_base)

dr, factor = 16, 4.0
original = build_inv_freq(dr)
linear = linear_pi_inv_freq(dr, 10_000.0, factor)
ntk = ntk_aware_inv_freq(dr, 10_000.0, factor)
print("pair | original | linear PI | NTK-aware")
for i in range(len(original)):
    print(f"{i:4d} | {original[i]:8.6f} | {linear[i]:9.6f} | {ntk[i]:9.6f}")
print("最高频 NTK/original:", (ntk[0] / original[0]).item())
print("最低频 NTK/original:", (ntk[-1] / original[-1]).item())


### Dynamic scaling 与 KV Cache 的一致性陷阱

如果同一会话越过原长度后突然更换 inv_freq，历史 K 已按旧频率旋转，而新 Q/K 用新频率，`R_m^T R_n = R_(n-m)` 的同频率前提被破坏。实现必须在 prefill 时选定频谱、重算历史 K，或严格采用框架规定的缓存策略。prefix cache 的 key 也应包含 scaling 配置。下面显示同一个历史 token 在不同 base 下会得到不同 K。


In [ ]:
history_k = torch.randn(1, 1, 1, dr)
history_pos = torch.tensor([[100]])
k_original = apply_rope_split(history_k, history_pos, original)
k_scaled = apply_rope_split(history_k, history_pos, ntk)
print("同一 K 更换频谱后的最大差异:", (k_original - k_scaled).abs().max().item())
assert not torch.allclose(k_original, k_scaled)


## 4. 左 padding、变长 batch、packing 与 cache offset

position id 是逻辑位置，不是 padded tensor 列号或物理 KV slot。左 padding 常由有效 mask 的累积和构造。独立样本 packing 时，通常同时使用 block-diagonal attention mask，并让每段 position 从 0 重置；连续对话前缀则不应按消息重置。prefill 有 S 个有效 token 后，首个 decode token 的 position 是 S。


In [ ]:
attention_mask = torch.tensor([[0, 0, 1, 1, 1], [1, 1, 1, 1, 1]], dtype=torch.long)
position_ids = attention_mask.cumsum(dim=-1) - 1
position_ids = position_ids.masked_fill(attention_mask == 0, 0)
print("attention mask:")
print(attention_mask)
print("position ids:")
print(position_ids)
assert position_ids[0].tolist() == [0, 0, 0, 1, 2]

valid_lengths = attention_mask.sum(dim=-1)
next_decode_positions = valid_lengths[:, None]
print("下一 decode token 的位置:", next_decode_positions.squeeze(-1).tolist())


## 5. BF16/FP16 相位精度

长位置先 cast 到低精度会丢失整数分辨率。BF16 指数范围大，但尾数短；相邻的大整数可能变成同一个值。稳健做法是在 autocast 外用 FP32 计算 `position × inv_freq` 和 sin/cos，再 cast 到 Q/K dtype。inv_freq 也应直接以高精度生成。


In [ ]:
large_positions = torch.tensor([100_000.0, 100_001.0])
as_bf16 = large_positions.to(torch.bfloat16)
print("FP32 positions:", large_positions.tolist())
print("BF16 positions:", as_bf16.tolist(), "是否折叠:", bool(as_bf16[0] == as_bf16[1]))
freq_fp32 = torch.outer(large_positions, build_inv_freq(8))
freq_bad = torch.outer(as_bf16, build_inv_freq(8).to(torch.bfloat16)).float()
print("相位生成最大差异:", (freq_fp32 - freq_bad).abs().max().item())


## 6. 二维与多模态 RoPE

图像 patch 有 `(row, col)` 两个坐标。轴向 2D RoPE 把 rotary 通道分成两组，分别按行和列旋转，使点积依赖 `(row2-row1, col2-col1)`，而不是只依赖展平索引。视频还可增加时间轴；多模态模型对文本/视觉的轴位置和后续 offset 各有约定。


In [ ]:
def apply_2d_rope(x, row_ids, col_ids, base=10_000.0):
    # x: [B,H,S,D]；前半通道给 row，后半给 col
    assert x.size(-1) % 4 == 0
    half = x.size(-1) // 2
    row = apply_rope_split(x[..., :half], row_ids, build_inv_freq(half, base))
    col = apply_rope_split(x[..., half:], col_ids, build_inv_freq(half, base))
    return torch.cat((row, col), dim=-1)

grid = torch.randn(1, 2, 6, 8)  # 2x3 patches
rows = torch.tensor([[0, 0, 0, 1, 1, 1]])
cols = torch.tensor([[0, 1, 2, 0, 1, 2]])
grid_rot = apply_2d_rope(grid, rows, cols)
print("2D RoPE input/output:", grid.shape, grid_rot.shape)
assert torch.allclose(grid.norm(dim=-1), grid_rot.norm(dim=-1), atol=1e-5)


## 7. 变体对比、工程坑与练习

| 方法 | 注入方式 | 扩长直觉 | 主要代价 |
|---|---|---|---|
| Learned absolute | embedding 相加 | 扩表/插值 | 表外位置未训练 |
| Relative bias / ALiBi | logits 加距离 bias | 分桶或线性距离 | 内容与位置较弱耦合 |
| RoPE | 旋转 Q/K | 多频率相对相位 | 相位 OOD、配置兼容 |
| PI | 所有频率统一降速 | 位置压回原窗口 | 近距分辨率下降 |
| NTK-aware / YaRN | 非均匀分频缩放 | 高频保局部、低频扩长 | 配置与训练更复杂 |

高频坑：split/interleaved 混用；cos/sin 广播轴错；position offset 错一位；K 重复旋转；动态 scaling 改频率却不重建历史 cache；左 padding 用列号；BF16 生成大位置相位；修改 base 后仍复用旧 prefix cache。

### 练习

1. 实现 interleaved 完整 apply 函数，并用维度置换证明它与 split-half 等价。
2. 实现 partial RoPE，使后半通道逐元素不变。
3. 比较 PI 与 NTK-aware 在距离 1、128、4096、16384 上的相位差。
4. 写 full-forward 与 cached-forward 的 RoPE logits 对齐测试。
5. 为 packed 独立序列构造 segment mask 与分段 position ids。

### 面试 60 秒主线

RoPE 把每对 Q/K 通道乘位置相关二维旋转；正交性保持范数，`R_m^T R_n=R_(n-m)` 让点积依赖相对位置。多频率兼顾局部与长距，但训练窗口外仍是分布外。PI 统一压缩位置，NTK/YaRN 类方法分频保留高频并拉伸低频。工程上最关键的是布局、FP32 相位、逻辑 position 与已旋转 K cache 的频率一致性。
